<a href="https://colab.research.google.com/github/h77k/python-ai-Trunova-Polina/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2b: Data Analysis — Чтение, очистка и подготовка выборок для визуализации

**Цель:**
Научиться читать CSV-файлы из вашего репозитория GitHub в Google Colab, выполнять базовую очистку данных с помощью pandas и готовить структурированные выборки для последующей визуализации в соответствии с требованиями курса.

**Данные:**
*   `data/models.csv` — информация о моделях техники (название, тип, производитель, страна, даты, габариты, масса и ссылки на изображения).
*   `data/manufacturers.csv` — информация о производителях (штаб-квартира, год основания, отрасль, веб-сайт).

**Что мы делаем:**
1.  Клонируем ваш репозиторий GitHub в Colab.
2.  Читаем CSV-файлы в pandas DataFrame.
3.  **Очистка и нормализация:**
    *   Переименовываем столбцы для удобства (сохраняем `URL`).
    *   Приводим числовые поля (`length`, `width`, `height`, `mass`) к числовому типу через `pd.to_numeric(errors='coerce')`.
    *   Извлекаем год начала производства (`start_year`) из даты.
4.  **Анализ заполненности:**
    *   Рассчитываем долю непустых значений для OPTIONAL-полей перед любым заполнением пропусков.
5.  **Формирование рабочих выборок:**
    *   Создаем отдельные датафреймы (`df_full`, `df_dims`, `df_time`, `df_type`) для разных типов визуализаций, чтобы четко понимать размер выборки (N) для каждого графика.

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем репозиторий проекта в Colab

import os

# Имя вашего репозитория
repo = "python-ai-Trunova-Polina"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

# Клонируем, если папки еще нет
if not os.path.exists(repo_path):
    !git clone -q https://github.com/h77k/python-ai-Trunova-Polina.git

# Переходим в папку репозитория, если мы еще не там
if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Trunova-Polina
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Trunova-Polina


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [2]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

# Читаем основной файл с моделями
df = pd.read_csv("data/models.csv")

print("✅ Загружено строк в df:", len(df))
df.head()

✅ Загружено строк в df: 1720


,model,modelLabel,type,typeLabel,manufacturer,manufacturerLabel,countryLabel,startProduction,predecessorLabel,successorLabel,length,width,height,mass,image
0,http://www.wikidata.org/entity/Q10716597,Volkswagen Auto 2000,http://www.wikidata.org/entity/Q47202313,Auto 2000,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
1,http://www.wikidata.org/entity/Q118870625,MPV Tehran,http://www.wikidata.org/entity/Q3882470,one-off vehicle,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
2,http://www.wikidata.org/entity/Q127327914,Volkswagen Beetle of José Mujica,http://www.wikidata.org/entity/Q152946,Volkswagen Käfer,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
3,http://www.wikidata.org/entity/Q109773142,Volkswagen Type 1 Sedan,http://www.wikidata.org/entity/Q152946,Volkswagen Käfer,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,http://www.wikidata.org/entity/Q108297293,Polo Flight,http://www.wikidata.org/entity/Q1420,автомобиль,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 🧹 [2B] Очистка и переименование столбцов

В исходном файле `models.csv` есть технические столбцы и поля, требующие нормализации для корректного анализа и визуализации.

**Что мы делаем в этом шаге:**

1.  **Переименование столбцов:**
    *   `model` (ссылка/идентификатор) → переименовываем в **`URL`**. Это прямое требование задания: сохранить ссылку на объект, но дать ей понятное имя.
    *   Столбцы с метками (`*Label`) переименовываем в читаемые названия на английском:
        *   `modelLabel` → `model_name`
        *   `typeLabel` → `type_name`
        *   `manufacturerLabel` → `manufacturer_name`
        *   `countryLabel` → `country_name`
        *   `startProduction` → `start_production` (оставляем как есть или упрощаем, далее преобразуем в год)

2.  **Приведение числовых типов:**
    *   Столбцы габаритов и массы: `length`, `width`, `height`, `mass`.
    *   Используем `pd.to_numeric(..., errors='coerce')`. Это превратит некорректные значения (например, строки или пустые места) в `NaN`, что позволит нам честно оценить заполненность данных.

3.  **Работа с датами:**
    *   Из столбца `start_production` извлекаем год (`start_year`). Это необходимо для построения временных рядов и графиков по десятилетиям.

4.  **Анализ заполненности (IMPORTANT):**
    *   Перед удалением пропусков мы обязаны посчитать долю заполненных значений для OPTIONAL-полей (`start_year`, габариты, предшественники/преемники). Это покажет, какие признаки реально можно использовать для визуализации, а какие слишком "дырявые".

⚠️ **Важно:** Мы не заменяем пропуски нулями (`fillna(0)`) на этом этапе, так как для физических величин (длина, масса) ноль — это некорректное значение. Мы будем формировать отдельные выборки (`dropna`) для каждого конкретного графика.

In [3]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# Проверка: нужна ли очистка? (если мы уже запускали эту ячейку, столбцы будут переименованы)
if "modelLabel" in df.columns:

    # 1. Переименование столбцов
    # URL сохраняем (требуется заданием), остальные метки делаем читаемыми
    df = df.rename(columns={
        'model': 'URL',                # Технический ID/ссылка -> URL
        'modelLabel': 'model_name',    # Название модели
        'typeLabel': 'type_name',      # Тип техники
        'manufacturerLabel': 'manufacturer_name', # Производитель
        'countryLabel': 'country_name', # Страна
        'startProduction': 'start_production'     # Дата начала производства
    })

    # 2. Приведение числовых полей к числовому типу
    # Используем errors='coerce', чтобы некорректные данные стали NaN (а не ошибкой)
    numeric_cols = ['length', 'width', 'height', 'mass']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Обработка даты: извлекаем год
    # Сначала преобразуем в datetime, потом берем .dt.year
    df['start_production'] = pd.to_datetime(df['start_production'], errors='coerce')
    df['start_year'] = df['start_production'].dt.year

    print("✅ DataFrame df очищен и нормализован")
else:
    print("⏭️ DataFrame df уже очищен, пропускаем шаг переименования")

print("\n✅ Данные готовы к анализу заполненности и формированию выборок")

✅ DataFrame df очищен и нормализован

✅ Данные готовы к анализу заполненности и формированию выборок


# 🔍 [3] Обзор данных: структура и первые строки

После первичной очистки сделаем короткий обзор нашего основного DataFrame `df`:

1.  Посмотрим размер таблицы (`shape`), чтобы понять объем данных.
2.  Выведем список столбцов и их типы (`dtypes`), чтобы убедиться, что числовые поля (`length`, `mass` и др.) действительно стали числами, а не остались строками.
3.  Посмотрим первые несколько строк (`head`), визуально оценив корректность переименования.
4.  Дополнительно посмотрим на статистическое описание (`describe`) для числовых колонок, чтобы выявить явные аномалии (например, отрицательную массу или нереалистичную длину).

Для удобства используем функцию `show_info`, адаптированную под одну таблицу.

In [4]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, типы данных и первые строки."""
    print(f"\n📊 {name}")
    print("-" * 30)
    print("Размер (строки, столбцы):", df.shape)
    print("\nТипы данных (dtypes):")
    print(df.dtypes)
    print("\nПервые строки:")
    display(df.head(n))  # display лучше форматирует таблицы в Colab

# 🔍 Шаг 3. Обзор данных

show_info(df, "Модели техники (df)")


📊 Модели техники (df)
------------------------------
Размер (строки, столбцы): (1720, 16)

Типы данных (dtypes):
URL                               object
model_name                        object
type                              object
type_name                         object
manufacturer                      object
manufacturer_name                 object
country_name                      object
start_production     datetime64[ns, UTC]
predecessorLabel                  object
successorLabel                    object
length                           float64
width                            float64
height                           float64
mass                             float64
image                             object
start_year                       float64
dtype: object

Первые строки:


,URL,model_name,type,type_name,manufacturer,manufacturer_name,country_name,start_production,predecessorLabel,successorLabel,length,width,height,mass,image,start_year
0,http://www.wikidata.org/entity/Q10716597,Volkswagen Auto 2000,http://www.wikidata.org/entity/Q47202313,Auto 2000,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
1,http://www.wikidata.org/entity/Q118870625,MPV Tehran,http://www.wikidata.org/entity/Q3882470,one-off vehicle,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
2,http://www.wikidata.org/entity/Q127327914,Volkswagen Beetle of José Mujica,http://www.wikidata.org/entity/Q152946,Volkswagen Käfer,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
3,http://www.wikidata.org/entity/Q109773142,Volkswagen Type 1 Sedan,http://www.wikidata.org/entity/Q152946,Volkswagen Käfer,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,http://www.wikidata.org/entity/Q108297293,Polo Flight,http://www.wikidata.org/entity/Q1420,автомобиль,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# ✅ [4] Быстрая проверка и валидация данных

На этом этапе мы проведем экспресс-анализ очищенных данных, чтобы понять их качество и содержание:

1.  **Категориальные данные:**
    *   Сколько уникальных производителей (`manufacturer_name`) и стран (`country_name`) представлено?
    *   Какие типы техники (`type_name`) встречаются чаще всего? (Это поможет понять структуру парка).
    *   Топ-5 стран и Топ-10 типов по количеству записей.

2.  **Временные данные:**
    *   Диапазон годов выпуска (`start_year`). Есть ли явные выбросы (например, будущие даты или слишком древние)?

3.  **Числовые данные (габариты и масса):**
    *   Базовая статистика (`describe`) для полей `length`, `width`, `height`, `mass`.
    *   Проверка на наличие отрицательных значений или нулей (которые могли возникнуть при ошибочной очистке, хотя мы использовали `coerce`).

4.  **Заполненность (Fill Rate):**
    *   Критически важный шаг: оценим долю заполненных значений для OPTIONAL-полей. Это покажет, какие признаки можно использовать для сквозного анализа, а для каких придется формировать отдельные подвыборки (например, только те модели, у которых известна масса).

Метод `.value_counts()` позволяет быстро оценить распределение категорий, а `.describe()` — статистические сводки по числам.

In [5]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных (df)")

# Создаем полную копию для общих проверок
df_full = df.copy()

# 1. Категориальный анализ
print("\n📊 Общие статистики:")
print(f"Уникальных моделей (строк): {len(df_full)}")
print(f"Уникальных производителей: {df_full['manufacturer_name'].nunique()}")
print(f"Уникальных стран: {df_full['country_name'].nunique()}")
print(f"Уникальных типов техники: {df_full['type_name'].nunique()}")

print("\n🏆 Топ-5 стран по числу моделей:")
print(df_full['country_name'].value_counts().head())

print("\n🚜 Топ-10 типов техники:")
print(df_full['type_name'].value_counts().head(10))

print("\n🏭 Топ-5 производителей:")
print(df_full['manufacturer_name'].value_counts().head())

# 2. Временной анализ
# Используем df_time (где год известен) для корректного диапазона
df_time = df_full.dropna(subset=['start_year']).copy()
if not df_time.empty:
    print(f"\n📅 Диапазон годов выпуска (для известных дат): {int(df_time['start_year'].min())} — {int(df_time['start_year'].max())}")
    print("\nСтатистика по годам выпуска:")
    print(df_time['start_year'].describe())
else:
    print("\n⚠️ Данные о годах выпуска отсутствуют или некорректны")

# 3. Числовой анализ (габариты и масса)
# Используем df_dims (где есть размеры) для релевантной статистики
df_dims = df_full.dropna(subset=['length', 'mass']).copy()
if not df_dims.empty:
    print("\n📏 Статистика по габаритам и массе (только для заполненных полей):")
    print(df_dims[['length', 'width', 'height', 'mass']].describe())
else:
    print("\n⚠️ Недостаточно данных о габаритах для статистики")

# 4. ОТЧЕТ ПО ЗАПОЛНЕННОСТИ (OPTIONAL FIELDS) - ВАЖНО ДЛЯ КУРСА
print("\n💧 Отчет по заполненности OPTIONAL-полей (доля непустых значений):")
optional_cols = ['start_year', 'length', 'width', 'height', 'mass', 'predecessorLabel', 'successorLabel', 'image']
# Фильтруем только те колонки, которые есть в датасете
existing_optional = [col for col in optional_cols if col in df_full.columns]
fill_rate = df_full[existing_optional].notna().mean().sort_values(ascending=False)
print(fill_rate)
print("\n❗ Обратите внимание: поля с низкой заполненностью потребуют формирования отдельных выборок для графиков.")

🔍 Быстрая проверка данных (df)

📊 Общие статистики:
Уникальных моделей (строк): 1720
Уникальных производителей: 6
Уникальных стран: 5
Уникальных типов техники: 114

🏆 Топ-5 стран по числу моделей:
country_name
Франция     672
США         584
Чехия       217
Россия      179
Германия     68
Name: count, dtype: int64

🚜 Топ-10 типов техники:
type_name
модель автомобиля                            1118
модель грузовика                               91
концепт-кар                                    72
модель гоночного автомобиля                    53
модель двигателя                               43
семейство двигателей                           41
модельный ряд автомобилей                      31
модель автобуса                                21
бензиновый двигатель внутреннего сгорания      14
модель боевой машины                           13
Name: count, dtype: int64

🏭 Топ-5 производителей:
manufacturer_name
Toyota                             672
Ford                               584
Šk

# 📝 Summary

Что мы сделали в этом ноутбуке (Week 2b):

✅ **Подготовка среды:** Клонировали ваш репозиторий `python-ai-Trunova-Polina` в Colab.
✅ **Чтение данных:** Загрузили `data/models.csv` в pandas DataFrame.
✅ **Очистка и нормализация:**
   *   Переименовали технические столбцы (`model` → `URL`, `*Label` → понятные имена).
   *   Привели числовые поля (`length`, `mass` и др.) к типу `float` через `pd.to_numeric(errors='coerce')`.
   *   Извлекли год производства (`start_year`) из даты.
✅ **Анализ качества данных:**
   *   Посчитали долю заполненности (Fill Rate) для OPTIONAL-полей.
   *   Выполнили быструю валидацию: топы стран, типов, производителей и статистику по габаритам.
✅ **Формирование рабочих выборок:**
   *   Подготовили логику для создания отдельных датафреймов (`df_full`, `df_dims`, `df_time`, `df_type`), чтобы избегать смешивания данных с пропусками и данных без них при построении графиков.

Теперь у нас есть чистая, структурированная база данных.
